# Claude Agent SDK: approvals and user input (`can_use_tool`, `AskUserQuestion`)

While working, Claude sometimes has to **check in with a human**:

1. **Permission**: "May I run this command / write this file?"
2. **Clarification**: "Which database should I use?" (via the `AskUserQuestion` tool)

Both are delivered to **one callback in your code**: `can_use_tool`. Execution **pauses** until your callback returns, and the callback can stay pending as long as it needs.

**What you will learn (and see running):**
1. When `can_use_tool` fires, and when it **never** does
2. The two answers it can return: **allow** and **deny**, and the six things you can *do* with them
3. A live approval flow where a denial teaches Claude a different approach
4. Handling clarifying questions from `AskUserQuestion`
5. What to do when the human takes a long time

> To keep the notebook runnable without you babysitting it, the approvals are **scripted by a policy function**. Flip one flag, `INTERACTIVE = True`, to answer the prompts yourself.

## The flow

```
 Claude wants to use a tool
          │
          ▼
  auto-approved by an allow rule or a mode (acceptEdits, bypassPermissions…)?
      ├── yes ──▶ tool runs.  can_use_tool is NEVER called
      └── no  ──▶  can_use_tool(tool_name, input_data, context)   ◀── your code
                         │
                         ├── PermissionResultAllow(updated_input=...)  → tool runs
                         └── PermissionResultDeny(message=...)         → Claude sees the message
```

`AskUserQuestion` takes the same path: it is just a tool whose name is `"AskUserQuestion"`.

**Warning:** because auto-approved tools skip the callback, **never use `can_use_tool` for rules that must apply to every call**. A `PreToolUse` **hook** runs *before* the rest of the permission flow and can allow, deny or modify anything.

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("approvals-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

In [ ]:
# OFFLINE
import re, json
from claude_agent_sdk import (
    query, ClaudeAgentOptions, HookMatcher,
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
    PermissionResultAllow, PermissionResultDeny,
)
from claude_agent_sdk.types import ToolPermissionContext, PermissionUpdate

INTERACTIVE = False     # True = you answer every prompt in the notebook; False = scripted policy

def short(value, n=140):
    text = str(value).replace("\n", " ")
    return text if len(text) <= n else text[: n - 1] + "…"

print("INTERACTIVE =", INTERACTIVE)

## 1. The callback signature

```python
async def can_use_tool(tool_name: str, input_data: dict, context: ToolPermissionContext):
    ...
    return PermissionResultAllow(updated_input=input_data)   # or PermissionResultDeny(message="...")
```

| Argument | What it is |
|---|---|
| `tool_name` | `"Bash"`, `"Write"`, `"Edit"`, `"Read"`, `"AskUserQuestion"`... |
| `input_data` | The tool's parameters. `Bash`: `command`, `description`, `timeout`. `Write`: `file_path`, `content`. `Edit`: `file_path`, `old_string`, `new_string`. `Read`: `file_path`, `offset`, `limit` |
| `context` | Extras: `suggestions` (ready-made permission updates), a cancellation `signal` (reserved in Python), plus tool-use metadata |

### Six things you can do with a request

| Response | How |
|---|---|
| **Approve** | `PermissionResultAllow(updated_input=input_data)` |
| **Approve with changes** | Allow with a **modified** `updated_input` (sanitize paths, add constraints). Claude is *not told* you changed anything |
| **Approve and remember** | Allow and echo a suggestion back in `updated_permissions` so future matching calls skip the prompt |
| **Reject** | `PermissionResultDeny(message="...")` |
| **Suggest an alternative** | Deny, with a message steering Claude to what the user wants instead |
| **Redirect entirely** | Send a brand-new instruction through **streaming input** |

When you **deny**, always write a useful message: Claude reads it and adjusts.

## 2. Write the policy (and test it offline)

Our scripted "human" follows these rules:

- `Bash` containing `rm` → **deny** and **suggest an alternative** (move files into a `trash` folder).
- Any other tool → **allow**.
- `AskUserQuestion` → collect answers (scripted: always pick the first option; or you, if `INTERACTIVE`).

We keep a log of every request so we can see exactly what reached the callback.

In [ ]:
# OFFLINE
approval_log = []     # (tool_name, short input) for every request that reaches the callback
questions_seen = []   # every question Claude asked


def parse_response(response, options):
    """Turn '1' or '1,3' into option labels; anything else is treated as free text."""
    try:
        indices = [int(s.strip()) - 1 for s in response.split(",")]
        labels = [options[i]["label"] for i in indices if 0 <= i < len(options)]
        return ", ".join(labels) if labels else response
    except ValueError:
        return response


def build_answers(input_data, chooser):
    """Build the structure AskUserQuestion expects: the original questions + an answers map."""
    answers = {q["question"]: chooser(q) for q in input_data.get("questions", [])}
    return {"questions": input_data.get("questions", []), "answers": answers}


def scripted_chooser(q):
    return q["options"][0]["label"]          # always the first option


def human_chooser(q):
    print(f"\n{q['header']}: {q['question']}")
    for i, opt in enumerate(q["options"]):
        print(f"  {i + 1}. {opt['label']} - {opt['description']}")
    print("  (number, comma-separated numbers, or your own text)" if q.get("multiSelect")
          else "  (a number, or your own text)")
    return parse_response(input("Your choice: ").strip(), q["options"])


async def can_use_tool(tool_name, input_data, context):
    approval_log.append((tool_name, short(input_data, 90)))

    # --- clarifying questions -------------------------------------------------
    if tool_name == "AskUserQuestion":
        questions_seen.extend(input_data.get("questions", []))
        chooser = human_chooser if INTERACTIVE else scripted_chooser
        return PermissionResultAllow(updated_input=build_answers(input_data, chooser))

    # --- tool approvals ----------------------------------------------------------
    if INTERACTIVE:
        print(f"\nTool: {tool_name}\nInput: {input_data}")
        if input("Allow this action? (y/n): ").lower() != "y":
            return PermissionResultDeny(message="User denied this action")
        return PermissionResultAllow(updated_input=input_data)

    if tool_name == "Bash" and re.search(r"\brm\b", input_data.get("command", "")):
        # Suggest an alternative: Claude reads this message and adapts.
        return PermissionResultDeny(
            message="User doesn't want files deleted. Move them into a folder named 'trash' instead."
        )
    return PermissionResultAllow(updated_input=input_data)


print("policy defined")

### Unit-test the callback with no API
It is just an async function, so we can call it with made-up requests and check its decisions.

In [ ]:
# OFFLINE
ctx = ToolPermissionContext(signal=None, suggestions=[])

async def decide(tool, data):
    result = await can_use_tool(tool, data, ctx)
    kind = "ALLOW" if isinstance(result, PermissionResultAllow) else "DENY "
    extra = result.message if isinstance(result, PermissionResultDeny) else ""
    return kind, extra

for tool, data in [
    ("Write", {"file_path": "a.txt", "content": "hi"}),
    ("Bash",  {"command": "ls -la"}),
    ("Bash",  {"command": "rm scratch.txt"}),
]:
    kind, extra = await decide(tool, data)
    print(f"{kind} {tool:6} {str(data)[:40]:42} {extra[:60]}")

print()
print("parse_response('2', ...)   ->", parse_response("2", [{"label": "A"}, {"label": "B"}]))
print("parse_response('1,2', ...) ->", parse_response("1,2", [{"label": "A"}, {"label": "B"}]))
print("parse_response('jquery')   ->", parse_response("jquery", [{"label": "A"}, {"label": "B"}]))

## 3. Live: approvals, and a denial that teaches Claude

Two details are needed in Python for `can_use_tool` to work:

1. The prompt must be a **streaming input** (an async generator), not a plain string.
2. A **dummy `PreToolUse` hook** that returns `{"continue_": True}`. The docs call this a *required workaround* that keeps the stream open for the callback.

Note we pass **no `allowed_tools`**, so every tool that needs approval reaches our callback.

In [ ]:
async def prompt_stream(text):
    yield {"type": "user", "message": {"role": "user", "content": text}}


async def keep_stream_open(input_data, tool_use_id, context):
    return {"continue_": True}      # required workaround for can_use_tool in Python


async def run_with_approvals(text, **opts):
    opts.setdefault("cwd", str(DEMO_DIR))
    options = ClaudeAgentOptions(
        can_use_tool=can_use_tool,
        hooks={"PreToolUse": [HookMatcher(matcher=None, hooks=[keep_stream_open])]},
        **opts,
    )
    try:
        async for m in query(prompt=prompt_stream(text), options=options):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, TextBlock):
                        print("  Claude:", short(b.text, 220))
                    elif isinstance(b, ToolUseBlock):
                        print(f"  [tool request] {b.name}: {short(b.input, 90)}")
            elif isinstance(m, ResultMessage):
                print(f"  [done: {m.subtype}]")
    except Exception as e:
        print("  (query raised:", short(e, 120), ")")

In [ ]:
import shutil
shutil.rmtree(DEMO_DIR / "trash", ignore_errors=True)
(DEMO_DIR / "scratch.txt").unlink(missing_ok=True)
approval_log.clear()

await run_with_approvals(
    "Create a file named scratch.txt containing the word hello. "
    "Then delete scratch.txt with the rm command."
)

print("\nEvery request that reached can_use_tool:")
for tool, data in approval_log:
    print(f"  {tool:6} {data}")
print("\nscratch.txt still exists?", (DEMO_DIR / "scratch.txt").exists())
print("trash folder:", sorted(p.name for p in (DEMO_DIR / "trash").glob("*")) if (DEMO_DIR / "trash").exists() else "none")

Read the log: the `rm` request was **denied** with our suggestion, and Claude (which saw the message as the tool result) came back with a different approach: moving the file into `trash/`. A rejection is not an error; it is *feedback*.

## 4. The callback does **not** fire for auto-approved tools

Same kind of task, but now `Read` is in `allowed_tools`, so it is pre-approved. Watch the log: **no `Read` request appears**.

In [ ]:
(DEMO_DIR / "hello.txt").write_text("The secret word is MANGO.\n")
approval_log.clear()

await run_with_approvals(
    "Read hello.txt and tell me the secret word.",
    allowed_tools=["Read"],
)

print("\nrequests that reached can_use_tool:", approval_log or "none (Read was auto-approved)")

**Takeaway:** an allow rule (or a mode like `acceptEdits` / `bypassPermissions`) resolves the call *before* `can_use_tool` is consulted. For logic that must see **every** call, use a `PreToolUse` hook instead.

Other notes:
- In `dontAsk` mode, anything that would prompt is simply **denied** and the callback is not used.
- The `PermissionRequest` hook can send Slack/email/push notifications when Claude is waiting for approval.

## 5. Approve with changes, and approve-and-remember

### Approve with changes
Allow the call but **modify the input** first. Claude is not told. Useful for sandboxing.

In [ ]:
# OFFLINE
async def sandboxing_callback(tool_name, input_data, context):
    if tool_name == "Bash":
        sandboxed = {**input_data, "command": input_data["command"].replace("/tmp", "/tmp/sandbox")}
        return PermissionResultAllow(updated_input=sandboxed)
    return PermissionResultAllow(updated_input=input_data)

result = await sandboxing_callback("Bash", {"command": "ls /tmp"}, ctx)
print("Claude asked :", "ls /tmp")
print("Actually runs:", result.updated_input["command"])

### Approve and remember
`context.suggestions` holds ready-made `PermissionUpdate` entries. Echo one back in `updated_permissions` and a suggestion whose `destination` is `"localSettings"` is written to `.claude/settings.local.json`, so **future sessions skip the prompt** for matching calls. (Needs `claude-agent-sdk` 0.1.80 or later.)

In [ ]:
# OFFLINE
async def remembering_callback(tool_name, input_data, context):
    choice = "always"     # a real UI would ask: once / always / no
    if choice == "always":
        persist = [s for s in context.suggestions if s.destination == "localSettings"]
        return PermissionResultAllow(updated_input=input_data, updated_permissions=persist)
    return PermissionResultAllow(updated_input=input_data)

fake_context = ToolPermissionContext(
    signal=None,
    suggestions=[
        PermissionUpdate(type="addRules", destination="session"),
        PermissionUpdate(type="addRules", destination="localSettings"),
    ],
)
r = await remembering_callback("Bash", {"command": "ls"}, fake_context)
print("rules persisted:", [(u.type, u.destination) for u in r.updated_permissions])

## 6. Clarifying questions: `AskUserQuestion`

When a task has several valid approaches, Claude may call the **`AskUserQuestion`** tool. It triggers the *same* `can_use_tool` callback with `tool_name == "AskUserQuestion"`.

- Claude **writes the questions and options**. You present them and return the user's picks. You **cannot add your own questions** to this flow.
- Each call has **1-4 questions**, each with **2-4 options** (`label`, `description`), a short `header` (max 12 chars) and a `multiSelect` flag.
- It is especially common in **`plan` mode**.
- If you pass a restrictive `tools=[...]` list, **include `"AskUserQuestion"`** or Claude cannot ask.
- **Not available in subagents.**

What the callback receives:

```json
{"questions": [{
   "question": "How should I format the output?",
   "header": "Format",
   "options": [{"label": "Summary", "description": "Brief overview"},
               {"label": "Detailed", "description": "Full explanation"}],
   "multiSelect": false}]}
```

What you return: the **original `questions`** plus an **`answers`** map from each *question text* to the chosen *label* (an array, or labels joined with `", "`, for multi-select; free text for an "Other" choice):

```python
PermissionResultAllow(updated_input={
    "questions": input_data["questions"],
    "answers": {"How should I format the output?": "Summary"},
})
```

Optional `response` carries a freeform reply when the user dismisses the card and types a general message instead (Claude then sees "The user responded: ...").

Test the answer builder offline with the exact sample above:

In [ ]:
# OFFLINE
sample = {"questions": [
    {"question": "How should I format the output?", "header": "Format",
     "options": [{"label": "Summary", "description": "Brief overview"},
                 {"label": "Detailed", "description": "Full explanation"}],
     "multiSelect": False},
    {"question": "Which sections should I include?", "header": "Sections",
     "options": [{"label": "Introduction", "description": "Opening context"},
                 {"label": "Conclusion", "description": "Final summary"}],
     "multiSelect": True},
]}

built = build_answers(sample, scripted_chooser)
print(json.dumps(built["answers"], indent=2))
print("questions passed through unchanged:", built["questions"] == sample["questions"])

### Live: make Claude ask us something
We instruct Claude to use `AskUserQuestion` first, so the demo is reliable. The scripted policy always chooses the **first option**.

In [ ]:
questions_seen.clear()
approval_log.clear()

await run_with_approvals(
    "Before doing anything else, you MUST call the AskUserQuestion tool to ask me which format "
    "I want for a one-line status report: 'Summary' or 'Detailed'. "
    "Then write one sentence in the format I chose, about the files in this folder.",
    allowed_tools=["Read", "Glob"],
)

print("\nquestions Claude asked:")
for q in questions_seen:
    print(f"  [{q['header']}] {q['question']}  options={[o['label'] for o in q['options']]}")

## 7. When the human is slow

The callback may stay pending **indefinitely**, but your process might not be able to. For long waits (a manager approving over Slack, say) use a `PreToolUse` hook that returns the **`defer`** decision instead of waiting inside the callback. The process can then exit, and **resume later from the persisted session** (see the sessions notebooks).

## 8. Other ways to get input

| Need | Use |
|---|---|
| Interrupt the agent mid-task, add context unprompted, build a chat UI | **Streaming input** |
| Forms, wizards, multi-step flows beyond multiple choice | **Custom tools** |
| Hook into an existing approval or ticketing system | **Custom tools** or a `PermissionRequest` hook |

## 9. Recap

- `can_use_tool(tool_name, input_data, context)` fires for **unapproved tool calls** and for **`AskUserQuestion`**. It pauses the run until it returns.
- Return **`PermissionResultAllow`** (optionally with modified input or remembered rules) or **`PermissionResultDeny(message=...)`**. Denials are feedback: Claude reads them and adapts.
- It **never fires** for auto-approved tools. Use a **`PreToolUse` hook** for universal rules.
- Python needs **streaming input** plus the **dummy hook** workaround.
- Clarifying questions: Claude writes them; you return `questions` + `answers` keyed by question text.
- For long waits, **defer** with a hook and resume the session later.

### Try it yourself
1. Set `INTERACTIVE = True`, re-run the demo, and deny the first tool call. How does Claude react?
2. Make the policy **approve with changes**: rewrite every `Write` path to live under `sandbox/`.
3. Switch to `permission_mode="plan"` and see how often Claude asks clarifying questions.
4. Add a `PreToolUse` hook that blocks `rm` even when `Bash` is in `allowed_tools`, and compare with the callback.
5. Next: **`agent-sdk-streaming-output-explained.ipynb`**.